# PROV-O → NetworkX 入門

- [PROV Model Primer](https://www.w3.org/TR/prov-primer/#intuitive-overview-of-prov)

## 準備

In [43]:
from rdflib import Graph, Dataset, Namespace
from rdflib.namespace import RDF, PROV
from rdflib.extras.external_graph_libs import rdflib_to_networkx_multidigraph
from yfiles_jupyter_graphs import GraphWidget, Layout
from IPython.display import display

import networkx as nx


def show_rdf_graph(rdf_graph: Graph) -> GraphWidget:
    """Visualize an RDFLib graph with yFiles."""
    nx_graph = rdflib_to_networkx_multidigraph(
        rdf_graph,
        edge_attrs=lambda s, p, o: {
            "key": p,
            "label": p.n3(rdf_graph.namespace_manager),
        },
    )
    widget = GraphWidget(
        graph=nx_graph,
        graph_layout=Layout.HIERARCHICAL,
    )
    return widget

W3C = Namespace("http://example.org#")

## RDFLib で PROV-O を読む

In [44]:
# W3C PROV-O Example 1 の主要部分を抜粋
prov_example = """
@prefix : <http://example.org#> .
@prefix prov: <http://www.w3.org/ns/prov#> .

:crimeData a prov:Entity .
:nationalRegionsList a prov:Entity .

:aggregationActivity
    a prov:Activity ;
    prov:used :crimeData, :nationalRegionsList ;
    prov:wasAssociatedWith :derek .

:aggregatedByRegions
    a prov:Entity ;
    prov:wasGeneratedBy :aggregationActivity .

:illustrationActivity
    a prov:Activity ;
    prov:used :aggregatedByRegions ;
    prov:wasAssociatedWith :derek .

:bar_chart
    a prov:Entity ;
    prov:wasGeneratedBy :illustrationActivity ;
    prov:wasDerivedFrom :aggregatedByRegions ;
    prov:wasAttributedTo :derek .

:derek a prov:Agent .
"""

w3c_graph = Graph().parse(
    data=prov_example,
    format="turtle",
)

print("Triple count:", len(w3c_graph))

Triple count: 16


In [45]:
show_rdf_graph(w3c_graph)


GraphWidget(layout=Layout(height='500px', width='100%'))

## 来歴を問い合わせる

In [46]:
# bar_chart は何から派生したか？
for source in w3c_graph.objects(
    W3C.bar_chart,
    PROV.wasDerivedFrom,
):
    print("Derived from:", source)

# 集計処理は何を使用したか？
for entity in w3c_graph.objects(
    W3C.aggregationActivity,
    PROV.used,
):
    print("Used:", entity)

Derived from: http://example.org#aggregatedByRegions
Used: http://example.org#crimeData
Used: http://example.org#nationalRegionsList


## 応用

### 世界の事実と来歴を分離

In [47]:
EX = Namespace("https://example.org/")

ds = Dataset()

observed = ds.graph(EX.observed)
inferred = ds.graph(EX.inferred)
provenance = ds.graph(EX.provenance)

# 観測・既知の情報
observed.add((EX.wallet, EX.onTopOf, EX.table))
observed.add((EX.table, EX.locatedIn, EX.kitchen))

# 推論済みと想定する情報
inferred.add((EX.wallet, EX.locatedIn, EX.kitchen))

print("Observed:", len(observed))
print("Inferred:", len(inferred))

Observed: 2
Inferred: 1


In [48]:
display(show_rdf_graph(observed))
display(show_rdf_graph(inferred))


GraphWidget(layout=Layout(height='500px', width='100%'))

GraphWidget(layout=Layout(height='500px', width='100%'))

### 事実を PROV Entity として識別

RDF Reification で，RDF triple に Entity ID を付ける

In [49]:
claims = {
    # 付与するIRI, (subject, predicate, object)
    EX.c_wallet_on_table: (EX.wallet, EX.onTopOf, EX.table),
    EX.c_table_in_kitchen: (EX.table, EX.locatedIn, EX.kitchen),
    EX.c_wallet_in_kitchen: (EX.wallet, EX.locatedIn, EX.kitchen),
}

for claim, (s, p, o) in claims.items():
    provenance.add((claim, RDF.type, PROV.Entity))
    provenance.add((claim, RDF.type, RDF.Statement))

    provenance.add((claim, RDF.subject, s))
    provenance.add((claim, RDF.predicate, p))
    provenance.add((claim, RDF.object, o))

print("Claim entities:", len(claims))

Claim entities: 3


In [50]:
show_rdf_graph(provenance)


GraphWidget(layout=Layout(height='500px', width='100%'))

### 推論の来歴を記述

In [51]:
# 推論活動(PROV.Activity)
provenance.add((EX.infer_room_1, RDF.type, PROV.Activity))

# 活動が使用した入力(PROV.used)
for source in (
    EX.c_wallet_on_table,
    EX.c_table_in_kitchen,
):
    provenance.add((EX.infer_room_1, PROV.used, source))

    # 生成物から入力への来歴関係
    provenance.add(
        (
            EX.c_wallet_in_kitchen,
            PROV.wasDerivedFrom,
            source,
        )
    )

# 推論結果の生成活動
provenance.add(
    (
        EX.c_wallet_in_kitchen,
        PROV.wasGeneratedBy,
        EX.infer_room_1,
    )
)

# 活動に関与したソフトウェア
provenance.add((EX.reasoner, RDF.type, PROV.Agent))
provenance.add(
    (
        EX.infer_room_1,
        PROV.wasAssociatedWith,
        EX.reasoner,
    )
)

print("Provenance triples:", len(provenance))

Provenance triples: 23


In [52]:
show_rdf_graph(provenance)


GraphWidget(layout=Layout(height='640px', width='100%'))

### NetworkX で依存関係を探索

In [53]:
# NetworkX で依存関係グラフを生成
dependency_graph = nx.DiGraph()

# 事実をノードとして登録
dependency_graph.add_nodes_from(claims)

# PROV-O の派生関係を反転
for derived, _, source in provenance.triples((None, PROV.wasDerivedFrom, None)):
    dependency_graph.add_edge(source, derived)

print("Nodes:", dependency_graph.number_of_nodes())
print("Edges:", dependency_graph.number_of_edges())

Nodes: 3
Edges: 2


In [ ]:
# 依存元から派生先へ向かう依存関係を可視化
GraphWidget(
    graph=dependency_graph,
    graph_layout=Layout.HIERARCHICAL,
)


GraphWidget(layout=Layout(height='500px', width='100%'))

In [54]:
# 直接依存する事実
print("Direct successors:")
for node in dependency_graph.successors(EX.c_wallet_on_table):
    print(node)

# 全ての下流ノード
print("\nDescendants:")
for node in nx.descendants(
    dependency_graph,
    EX.c_wallet_on_table,
):
    print(node)

# 結果に至る上流ノード
print("\nAncestors:")
for node in nx.ancestors(
    dependency_graph,
    EX.c_wallet_in_kitchen,
):
    print(node)


Direct successors:
https://example.org/c_wallet_in_kitchen

Descendants:
https://example.org/c_wallet_in_kitchen

Ancestors:
https://example.org/c_wallet_on_table
https://example.org/c_table_in_kitchen


### DAG 判定と循環の検出

In [56]:
print("Is DAG:", nx.is_directed_acyclic_graph(dependency_graph))

# コピーで循環を作る
cyclic_graph = dependency_graph.copy()

cyclic_graph.add_edge(
    EX.c_wallet_in_kitchen,
    EX.c_wallet_on_table,
)

print("After adding a cycle:", nx.is_directed_acyclic_graph(cyclic_graph))

Is DAG: True
After adding a cycle: False


In [57]:
GraphWidget(
    graph=cyclic_graph,
    graph_layout=Layout.HIERARCHICAL,
)


GraphWidget(layout=Layout(height='500px', width='100%'))